# Basics &mdash; Negating Implication, and Moving Terms Across It

**Concept 9 of the Basics decomposition:** *Negating Implication, and Moving Terms Across It*

$\neg(a\Rightarrow b)\equiv(a\wedge\neg b)$; and conjuncts can be shuffled across the arrow.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Negating-Implication/Concept-Negating-Implication.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Basics/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    # Clone if absent; PULL if already there. A Colab session reuses /content,
    # so without the pull you keep whatever was cloned earlier in the session.
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove;       else git -C Jove pull -q --ff-only 2>/dev/null || echo "(kept existing Jove clone)"; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import jove, os
# jove has no __init__.py, so it is a NAMESPACE package and jove.__file__ is
# None; the directory has to come from __path__.  Everything here is inside
# the try, so a failed self-check can never break the setup cell.
try:
    _p = list(jove.__path__)[0]
    _ok = 'toolbar stylesheet' in open(os.path.join(_p, 'AnimateDFA.py')).read()
    print("Jove loaded from", _p)
    print("animation toolbar fix:",
          'present' if _ok else 'MISSING -- re-clone or pull Jove')
except Exception as _e:
    print("Jove loaded (self-check unavailable:", _e, ")")

## 1. The idea


$$\neg(a \Rightarrow b) \ \equiv\ (a \wedge \neg b)$$

To refute an implication you must exhibit the antecedent **holding** while the
consequent **fails**. There is no other way, and no $\vee$ appears.

Terms can also be **moved across the arrow**, which is the manipulation the book
actually uses:

$$(a \wedge b) \Rightarrow c \ \equiv\ a \Rightarrow (b \Rightarrow c)$$
$$a \Rightarrow (b \vee c) \ \equiv\ (a \wedge \neg b) \Rightarrow c$$

The first is **currying** (Chapter 18!). The second is how a disjunctive goal becomes
an extra hypothesis &mdash; "to prove $b$ or $c$, assume $\neg b$ and prove $c$".

## 2. Definitions

### The laws

In [ ]:
IMP = lambda a, b: (not a) or b
B3 = [(a, b, c) for a in (False, True) for b in (False, True) for c in (False, True)]

def equiv3(f, g):
    return all(f(a, b, c) == g(a, b, c) for a, b, c in B3)

## 3. Tests

**Negating an implication** leaves a conjunction.

In [ ]:
for a in (False, True):
    for b in (False, True):
        print("  !(%-6s => %-6s) = %-6s   %-6s and !%-6s = %s"
              % (a, b, not IMP(a, b), a, b, a and not b))
assert all((not IMP(a, b)) == (a and not b)
           for a in (False, True) for b in (False, True))
print("\nNo 'or' anywhere: refuting a => b needs a TRUE and b FALSE.")

**Currying:** $(a\wedge b)\Rightarrow c \equiv a\Rightarrow(b\Rightarrow c)$.

In [ ]:
f = lambda a, b, c_: IMP(a and b, c_)
g = lambda a, b, c_: IMP(a, IMP(b, c_))
assert equiv3(f, g)
print("%-7s %-7s %-7s %-16s %s" % ("a", "b", "c", "(a and b) => c", "a => (b => c)"))
for a, b, c_ in B3:
    print("%-7s %-7s %-7s %-16s %s" % (a, b, c_, f(a, b, c_), g(a, b, c_)))
print("\nThis is currying -- the same law as Chapter 18, Concept 2.")

**A disjunctive goal becomes an extra hypothesis.**

In [ ]:
f = lambda a, b, c_: IMP(a, b or c_)
g = lambda a, b, c_: IMP(a and not b, c_)
assert equiv3(f, g)
print("  a => (b or c)   ==   (a and !b) => c   ?", equiv3(f, g))
print()
print("Reading: to prove 'b or c' from a, you may ASSUME !b and prove c.")
print("That is the standard move in a proof by cases.")

Two more shuffles worth knowing.

In [ ]:
pairs = [("a => (b and c)", lambda a, b, c_: IMP(a, b and c_),
          "(a=>b) and (a=>c)", lambda a, b, c_: IMP(a, b) and IMP(a, c_)),
         ("(a or b) => c",  lambda a, b, c_: IMP(a or b, c_),
          "(a=>c) and (b=>c)", lambda a, b, c_: IMP(a, c_) and IMP(b, c_))]
for n1, f1, n2, f2 in pairs:
    print("  %-18s ==  %-20s ? %s" % (n1, n2, equiv3(f1, f2)))
    assert equiv3(f1, f2)

A worked refutation, in the shape the book uses.

In [ ]:
# claim: for all n, (n is even) => (n is divisible by 4)   -- FALSE
claim = lambda n: IMP(n % 2 == 0, n % 4 == 0)
bad = [n for n in range(20) if not claim(n)]
print("counterexamples :", bad)
n = bad[0]
print("  n = %d : antecedent (even) = %s, consequent (div by 4) = %s"
      % (n, n % 2 == 0, n % 4 == 0))
assert n % 2 == 0 and n % 4 != 0
print("\nExactly the shape !(a => b) = a and !b demands.")

## 4. Exercises


1. Negate $(a \vee b) \Rightarrow (c \wedge d)$ down to literals.
2. Is $(a \Rightarrow b) \Rightarrow c$ the same as $a \Rightarrow (b \Rightarrow c)$?
3. Use the disjunctive-goal law on a proof you have written.

In [ ]:
# Your work for the exercises above.